# Frozen prospective claims: genes the glucose screen did not measure


**What this is.** A ranking of the genes the glucose-withdrawal screen did not measure, by how likely each is to be needed for growth without glucose -- written down, hashed and dated now, so a future screen can grade it without anyone having been able to adjust it (instruction 63).

**How it was made.** The reference strategy of the harness (boosted trees over every allowed evidence family; every pooled fitness screen held out, as the question requires) fitted on all 7,393 measured genes, then applied to the unmeasured ones. No frozen test partition is touched: this is the deployment fit, and the strategy's held-out record is in `results/borrowed_strategies_2026_11_06/` (frozen-test AUC 0.824).

**What it is not.** A fill. No strategy clears the 0.90 bar on this question, so these are predictions to check, never facts in the map.

Frozen 2026-11-06.

In [1]:
import hashlib, json, os, time
import numpy as np, pandas as pd
from parascape import harness as H, question_registry as Q, datasets as ds, organisms as O
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
question = Q.get('required_without_glucose')
measured = nodes['fit_no_glucose'].notna()
labels = (nodes['fit_no_glucose'] <= -2.0)[measured]
def family_of(column):
    found = ds.provenance(column, O.TOXOPLASMA)
    return found.key if found is not None else 'unregistered'
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
families = pd.Series({c: family_of(c) for c in numeric})
features = H.allowed_features(question, nodes[numeric].astype('float64'), families)
print(int(measured.sum()), 'measured genes;', int((~measured).sum()), 'unmeasured;',
      features.shape[1], 'allowed columns')

7393 measured genes; 747 unmeasured; 366 allowed columns


In [2]:
strategy = H.Strategy.of('boosted trees (reference)', 'hist_gbt', max_iter=150)
usable = [c for c in features.columns if features.loc[labels.index, c].notna().any()]
model = strategy.build().fit(features.loc[labels.index, usable], labels.astype(int))
unmeasured = nodes.index[~measured]
p = model.predict_proba(features.loc[unmeasured, usable])[:, 1]
claims = pd.DataFrame({'gene_id': unmeasured, 'question': question.id,
                       'probability_needed': np.round(p, 4)}).sort_values('probability_needed', ascending=False)
claims['rank'] = np.arange(1, len(claims) + 1)
claims['product'] = nodes.loc[claims.gene_id, 'product'].values if 'product' in nodes else ''
claims.head(15)

,gene_id,question,probability_needed,rank,product
216,TGME49_249180,required_without_glucose,0.8582,1,bifunctional dihydrofolate reductase-thymidylate synthase
304,TGME49_223150,required_without_glucose,0.3912,2,START domain-containing protein
259,TGME49_268830,required_without_glucose,0.3865,3,ATP synthase subunit delta
65,TGME49_500302,required_without_glucose,0.3656,4,Porphobilinogen Synthase (PBGS) / Aminolevulinic Acid Dehydratase (ALAD) + Pitrilysin (apicoplast stromal processing peptidase)
119,TGME49_500116,required_without_glucose,0.2906,5,"aspartyl/glutamyl-tRNA(Asn/Gln) amidotransferase subunit B, putative"
3,TGME49_500142,required_without_glucose,0.2903,6,"serine-protein kinase ATM, putative"
230,TGME49_500306,required_without_glucose,0.2876,7,adaptor complexes medium subunit family protein
17,TGME49_500124,required_without_glucose,0.2728,8,anaphase-promoting complex subunit 1
251,TGME49_500308,required_without_glucose,0.2697,9,"phosphatidate cytidylyltransferase, putative"
4,TGME49_500287,required_without_glucose,0.2552,10,"phosphatidylinositol 4-kinase, putative"


## The freeze

The claims file, its SHA-256, the strategy and evidence hashes, the version and the date go into `manifest.json`. Grading later means: recompute the hash, confirm it matches, then score the ranking against the new screen -- precision in the top 50, 100 and 200, and AUC.

In [3]:
from parascape import __version__
path = os.path.join(OUT, 'claims.tsv')
claims.to_csv(path, sep='\t', index=False)
digest = hashlib.sha256(open(path, 'rb').read()).hexdigest()
manifest = dict(question=question.id, label=question.label_source, frozen_at=time.strftime('%Y-%m-%dT%H:%M:%S'),
                parascape_version=__version__, strategy=strategy.name, strategy_hash=strategy.identity,
                evidence_hash=H.evidence_identity(question, features), n_training=int(len(labels)),
                n_training_positive=int(labels.sum()), n_claims=int(len(claims)),
                claims_sha256=digest, held_out_record='results/borrowed_strategies_2026_11_06/scores.tsv',
                grade_with='precision in the top 50/100/200 and AUC against the next genome-wide '
                           'glucose-withdrawal (or glycolysis-null) screen covering these genes')
json.dump(manifest, open(os.path.join(OUT, 'manifest.json'), 'w'), indent=1)
manifest

{'question': 'required_without_glucose', 'label': 'nodes.parquet:fit_no_glucose <= -2.0', 'frozen_at': '2026-11-06T22:46:21', 'parascape_version': '0.0.10.0', 'strategy': 'boosted trees (reference)', 'strategy_hash': '1b4ace1f7077151fe1eba96e64d52d1860b359f529b4c92f4bb2ea09e8c42901', 'evidence_hash': '5b67dd8831833c612bc10cf5a1742b5ec69758bc364e8c0eb6a0a9756a8b0499', 'n_training': 7393, 'n_training_positive': 1765, 'n_claims': 747, 'claims_sha256': '6b482f1023568d8bf339621502b160aa4a91a594c95330d800aed00b02695138', 'held_out_record': 'results/borrowed_strategies_2026_11_06/scores.tsv', 'grade_with': 'precision in the top 50/100/200 and AUC against the next genome-wide glucose-withdrawal (or glycolysis-null) screen covering these genes'}